# Crypto Lake — Data Audit

Verify completeness, bar intervals, gaps, and source distribution across all collected symbols.

**How to use:**
1. Set the date range in the **Parameters** cell below
2. Run All Cells (`Kernel → Restart & Run All`)
3. Each section is independent — run individual cells to re-check specific things

All queries run directly against parquet files via DuckDB — no database server needed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()) + '/notebooks')
import lake
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import date

pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.1f}'.format)

con = lake.connect()
print('DuckDB connected. Data root:', lake.DATA_ROOT)

## Parameters
Change these to adjust the audit date range.

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────
AUDIT_START = '2026-03-21'
AUDIT_END   = '2026-04-10'
# ────────────────────────────────────────────────────────────────────────────
print(f'Audit range: {AUDIT_START} → {AUDIT_END}')

---
## 1. Full Audit Summary
One row per symbol — total bars, gaps, and live vs backfill split.

In [ ]:
summary = lake.full_audit_summary(con, AUDIT_START, AUDIT_END)

# Styled display
def colour_gaps(val):
    if isinstance(val, (int, float)) and val > 0:
        return 'color: red; font-weight: bold'
    return ''

def colour_pct(val):
    if isinstance(val, float):
        if val < 50: return 'color: red'
        if val < 80: return 'color: orange'
    return ''

summary.style \
    .applymap(colour_gaps, subset=['total_gaps']) \
    .format({'total_bars': '{:,}', 'live_pct': '{:.1f}%', 'backfill_pct': '{:.1f}%'})

---
## 2. Per-Day Completeness — Choose a Symbol

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────
EXCHANGE = 'binance'
SYMBOL   = 'BTCUSDT'
# ────────────────────────────────────────────────────────────────────────────

df = lake.completeness_report(con, EXCHANGE, SYMBOL, AUDIT_START, AUDIT_END)
print(f'{EXCHANGE}/{SYMBOL}  ({lake.EXCHANGE_META[EXCHANGE]["interval_sec"]}s bars)')

df.style \
    .applymap(lambda v: 'color: red'    if v == 'MISSING' else
                        'color: orange' if v == 'LOW'     else '', subset=['status']) \
    .applymap(lambda v: 'color: red; font-weight: bold' if isinstance(v,(int,float)) and v > 0 else '',
              subset=['gaps']) \
    .format({'bar_count': '{:,}', 'live': '{:,}', 'backfill': '{:,}',
             'empty': '{:,}', 'completeness_pct': '{:.1f}%'})

In [ ]:
# Bar count chart
fig, ax = plt.subplots(figsize=(14, 4))
interval_sec = lake.EXCHANGE_META[EXCHANGE]['interval_sec']
expected = 86400 // interval_sec
df_plot = df[df['bar_count'] > 0]
colours = ['steelblue' if s == 'OK' else 'orange' if s == 'partial' else 'red'
           for s in df_plot['status']]
ax.bar(df_plot.index, df_plot['bar_count'], color=colours, label='bars collected')
ax.axhline(expected, color='green', linestyle='--', linewidth=1, label=f'Expected ({expected:,}/day)')
ax.set_title(f'{EXCHANGE}/{SYMBOL} — daily bar count')
ax.set_ylabel('bars')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Gap Analysis

In [ ]:
gaps = lake.find_gaps(con, EXCHANGE, SYMBOL, AUDIT_START, AUDIT_END)

if gaps.empty:
    print(f'✓ No gaps found in {EXCHANGE}/{SYMBOL} between {AUDIT_START} and {AUDIT_END}')
else:
    print(f'{len(gaps)} gap(s) found in {EXCHANGE}/{SYMBOL}:')
    display(gaps.style.format({'duration_sec': '{:,}'}))

In [ ]:
# Quick gap check for ALL symbols
print('Gap summary across all symbols (showing only symbols with gaps)\n')

for exchange, meta in lake.EXCHANGE_META.items():
    for symbol in meta['symbols']:
        g = lake.find_gaps(con, exchange, symbol, AUDIT_START, AUDIT_END)
        if not g.empty:
            total_sec = g['duration_sec'].sum()
            print(f'  {exchange}/{symbol}: {len(g)} gap(s), total lost = {lake._fmt_dur(int(total_sec))}')
            for _, row in g.iterrows():
                print(f'    {row["gap_start"].strftime("%Y-%m-%d %H:%M")} → '
                      f'{row["gap_end"].strftime("%H:%M")}  ({row["duration_human"]})')

---
## 4. Bar Interval Verification
Confirm Binance bars are 1s and Coinbase/Kraken are 60s.

In [ ]:
from collections import Counter
CHECK_DATE = '2026-04-09'

print(f'Bar interval check on {CHECK_DATE}\n')
print(f'{"Exchange":<12} {"Symbol":<12} {"Expected":>10} {"Actual mode":>12} {"Status"}')
print('-'*60)

for exchange, meta in lake.EXCHANGE_META.items():
    expected_iv = meta['interval_sec']
    symbol = meta['symbols'][0]
    df_check = lake.query_symbol(con, exchange, symbol, CHECK_DATE, CHECK_DATE)
    if df_check.empty:
        print(f'{exchange:<12} {symbol:<12} {expected_iv:>10}s  {"no data":>12}')
        continue
    ts = df_check.index.astype('int64') // 10**9  # to unix seconds
    diffs = ts.diff().dropna().astype(int).tolist()
    mode_iv, mode_count = Counter(diffs).most_common(1)[0]
    status = 'OK' if mode_iv == expected_iv else f'MISMATCH (got {mode_iv}s)'
    print(f'{exchange:<12} {symbol:<12} {expected_iv:>10}s  {mode_iv:>11}s  {status}  (n={mode_count}/{len(diffs)})')

---
## 5. Source Distribution
How much data is live vs backfilled vs empty bars?

In [ ]:
# Distribution for selected symbol
src = lake.source_distribution(con, EXCHANGE, SYMBOL, AUDIT_START, AUDIT_END)
total = src['count'].sum()
src['pct'] = (src['count'] / total * 100).round(1)
print(f'Source distribution for {EXCHANGE}/{SYMBOL}')
display(src)

fig, ax = plt.subplots(figsize=(6, 4))
ax.pie(src['count'], labels=src['source'], autopct='%1.1f%%',
       colors=['steelblue','tomato','orange','grey'][:len(src)])
ax.set_title(f'{EXCHANGE}/{SYMBOL} source mix')
plt.tight_layout()
plt.show()

---
## 6. Price Chart
OHLC close-line chart for any symbol and date range.

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────
CHART_EXCHANGE = 'binance'
CHART_SYMBOL   = 'BTCUSDT'
CHART_START    = '2026-04-07'
CHART_END      = '2026-04-10'
RESAMPLE_RULE  = '1min'  # e.g. '1min', '5min', '1h', '1D'
# ────────────────────────────────────────────────────────────────────────────

df_c = lake.query_symbol(con, CHART_EXCHANGE, CHART_SYMBOL, CHART_START, CHART_END,
                         cols='window_start, open, high, low, close, volume_base, source')

if df_c.empty:
    print('No data')
else:
    # Resample to requested rule
    ohlcv = df_c[['open','high','low','close','volume_base']].resample(RESAMPLE_RULE).agg(
        open=('open','first'), high=('high','max'), low=('low','min'),
        close=('close','last'), volume=('volume_base','sum')
    ).dropna()

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 7), sharex=True,
                                    gridspec_kw={'height_ratios': [3,1]})
    ax1.plot(ohlcv.index, ohlcv['close'], linewidth=0.8, color='steelblue')
    ax1.fill_between(ohlcv.index, ohlcv['low'], ohlcv['high'], alpha=0.15, color='steelblue')
    ax1.set_title(f'{CHART_EXCHANGE}/{CHART_SYMBOL}  {RESAMPLE_RULE} candles  {CHART_START}→{CHART_END}')
    ax1.set_ylabel('Price (USD)')
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax1.grid(True, alpha=0.3)

    ax2.bar(ohlcv.index, ohlcv['volume'], width=0.001*len(ohlcv), color='steelblue', alpha=0.6)
    ax2.set_ylabel('Volume')
    ax2.grid(True, alpha=0.3)

    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
    print(f'Rows: {len(ohlcv):,}  |  Price range: ${ohlcv["low"].min():,.2f} – ${ohlcv["high"].max():,.2f}')

---
## 7. Raw Data Explorer
Load any slice of raw bars into a DataFrame for custom analysis.

In [ ]:
# Load a day of raw bars
df_raw = lake.query_symbol(con, 'binance', 'BTCUSDT', '2026-04-09', '2026-04-09')
print(f'Rows: {len(df_raw):,}  |  Columns: {list(df_raw.columns)}')
df_raw.head(10)

In [ ]:
# Custom SQL — run any DuckDB query directly
# Example: average trade count per hour for BTCUSDT on a specific day
from pathlib import Path
from datetime import date

d = date(2026, 4, 9)
dp = lake.DATA_ROOT / 'binance' / 'BTCUSDT' / f'year={d.year}' / f'month={d.month:02d}' / f'day={d.day:02d}'
glob = str(dp).replace('\\', '/') + '/*.parquet'

result = con.execute(f"""
    SELECT
        strftime(window_start AT TIME ZONE 'UTC', '%H') AS hour_utc,
        COUNT(*) AS bar_count,
        SUM(trade_count) AS total_trades,
        AVG(close) AS avg_close,
        SUM(volume_base) AS volume
    FROM read_parquet('{glob}')
    GROUP BY 1
    ORDER BY 1
""").df()

result.style.format({'bar_count': '{:,}', 'total_trades': '{:,.0f}',
                     'avg_close': '${:,.2f}', 'volume': '{:,.4f}'})

---
## 8. Disk Usage

In [ ]:
total = sum(f.stat().st_size for f in lake.DATA_ROOT.rglob('*.parquet'))
file_count = sum(1 for _ in lake.DATA_ROOT.rglob('*.parquet'))
jsonl_root = lake.DATA_ROOT.parent / 'jsonl'
jsonl_bytes = sum(f.stat().st_size for f in jsonl_root.rglob('*') if f.is_file()) if jsonl_root.exists() else 0

print(f'Parquet files : {file_count:,}')
print(f'Parquet size  : {total / 1024**3:.3f} GB  ({total / 1024**2:.1f} MB)')
if jsonl_bytes:
    print(f'JSONL size    : {jsonl_bytes / 1024**3:.3f} GB  ({jsonl_bytes / 1024**2:.1f} MB)')
print(f'Grand total   : {(total + jsonl_bytes) / 1024**3:.3f} GB')

# Per-exchange breakdown
print()
for ex_path in sorted(lake.DATA_ROOT.iterdir()):
    ex_bytes = sum(f.stat().st_size for f in ex_path.rglob('*.parquet'))
    ex_files = sum(1 for _ in ex_path.rglob('*.parquet'))
    print(f'  {ex_path.name:<12} {ex_bytes/1024**2:>8.1f} MB  ({ex_files:,} files)')